In [0]:
spark

In [0]:
# Project: Databricks NYC Taxi Lakehouse Analytics
# Notebook: 02_data_cleaning_silver
# Objective: Clean Bronze data and create Silver Delta tables

from pyspark.sql import functions as F

bronze_taxi_table = "workspace.default.bronze_yellow_taxi_trips"
bronze_zones_table = "workspace.default.bronze_taxi_zones"

df_taxi_bronze = spark.table(bronze_taxi_table)
df_zones_bronze = spark.table(bronze_zones_table)

print("Bronze tables loaded successfully.")
print("Taxi rows:", df_taxi_bronze.count())
print("Zones rows:", df_zones_bronze.count())

Bronze tables loaded successfully.
Taxi rows: 2964624
Zones rows: 265


In [0]:
# Initial data quality checks - Bronze taxi data

total_rows = df_taxi_bronze.count()

null_check = df_taxi_bronze.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_taxi_bronze.columns
])

print("Total rows:", total_rows)

display(null_check)

Total rows: 2964624


VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,0,0,140162,0,140162,140162,0,0,0,0,0,0,0,0,0,0,140162,140162


In [0]:
# Create cleaned Silver dataframe

df_taxi_silver = (
    df_taxi_bronze
    .filter(F.col("tpep_pickup_datetime").isNotNull())
    .filter(F.col("tpep_dropoff_datetime").isNotNull())
    .filter(F.col("trip_distance") > 0)
    .filter(F.col("fare_amount") > 0)
    .filter(F.col("total_amount") > 0)
    .filter(F.col("PULocationID").isNotNull())
    .filter(F.col("DOLocationID").isNotNull())
    .withColumn(
        "trip_duration_minutes",
        (
            F.unix_timestamp("tpep_dropoff_datetime") -
            F.unix_timestamp("tpep_pickup_datetime")
        ) / 60
    )
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn("pickup_year", F.year("tpep_pickup_datetime"))
    .withColumn("pickup_month", F.month("tpep_pickup_datetime"))
)

print("Silver dataframe created.")
print("Bronze rows:", df_taxi_bronze.count())
print("Silver rows:", df_taxi_silver.count())

Silver dataframe created.
Bronze rows: 2964624
Silver rows: 2869714


In [0]:
# Preview Silver dataframe with selected columns

display(
    df_taxi_silver.select(
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "trip_duration_minutes",
        "pickup_date",
        "pickup_hour",
        "pickup_year",
        "pickup_month",
        "trip_distance",
        "fare_amount",
        "total_amount",
        "PULocationID",
        "DOLocationID"
    ).limit(10)
)

tpep_pickup_datetime,tpep_dropoff_datetime,trip_duration_minutes,pickup_date,pickup_hour,pickup_year,pickup_month,trip_distance,fare_amount,total_amount,PULocationID,DOLocationID
2024-01-01T00:57:55.000,2024-01-01T01:17:43.000,19.8,2024-01-01,0,2024,1,1.72,17.7,22.7,186,79
2024-01-01T00:03:00.000,2024-01-01T00:09:36.000,6.6,2024-01-01,0,2024,1,1.8,10.0,18.75,140,236
2024-01-01T00:17:06.000,2024-01-01T00:35:01.000,17.916666666666668,2024-01-01,0,2024,1,4.7,23.3,31.3,236,79
2024-01-01T00:36:38.000,2024-01-01T00:44:56.000,8.3,2024-01-01,0,2024,1,1.4,10.0,17.0,79,211
2024-01-01T00:46:51.000,2024-01-01T00:52:57.000,6.1,2024-01-01,0,2024,1,0.8,7.9,16.1,211,148
2024-01-01T00:54:08.000,2024-01-01T01:26:31.000,32.38333333333333,2024-01-01,0,2024,1,4.7,29.6,41.5,148,141
2024-01-01T00:49:44.000,2024-01-01T01:15:47.000,26.05,2024-01-01,0,2024,1,10.82,45.7,64.95,138,181
2024-01-01T00:30:40.000,2024-01-01T00:58:40.000,28.0,2024-01-01,0,2024,1,3.0,25.4,30.4,246,231
2024-01-01T00:26:01.000,2024-01-01T00:54:12.000,28.183333333333334,2024-01-01,0,2024,1,5.44,31.0,36.0,161,261
2024-01-01T00:28:08.000,2024-01-01T00:29:16.000,1.1333333333333333,2024-01-01,0,2024,1,0.04,3.0,8.0,113,113


In [0]:
# Save Silver Delta table

silver_taxi_table = "workspace.default.silver_yellow_taxi_trips"

df_taxi_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(silver_taxi_table)

print("Silver table created successfully.")

Silver table created successfully.


In [0]:
# Validate Silver Delta table

silver_taxi_count = spark.table("workspace.default.silver_yellow_taxi_trips").count()

print("Silver taxi rows:", silver_taxi_count)

display(
    spark.table("workspace.default.silver_yellow_taxi_trips")
    .select(
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "trip_duration_minutes",
        "pickup_date",
        "pickup_hour",
        "pickup_year",
        "pickup_month",
        "trip_distance",
        "fare_amount",
        "total_amount",
        "PULocationID",
        "DOLocationID"
    )
    .limit(10)
)

Silver taxi rows: 2869714


tpep_pickup_datetime,tpep_dropoff_datetime,trip_duration_minutes,pickup_date,pickup_hour,pickup_year,pickup_month,trip_distance,fare_amount,total_amount,PULocationID,DOLocationID
2024-01-01T00:57:55.000,2024-01-01T01:17:43.000,19.8,2024-01-01,0,2024,1,1.72,17.7,22.7,186,79
2024-01-01T00:03:00.000,2024-01-01T00:09:36.000,6.6,2024-01-01,0,2024,1,1.8,10.0,18.75,140,236
2024-01-01T00:17:06.000,2024-01-01T00:35:01.000,17.916666666666668,2024-01-01,0,2024,1,4.7,23.3,31.3,236,79
2024-01-01T00:36:38.000,2024-01-01T00:44:56.000,8.3,2024-01-01,0,2024,1,1.4,10.0,17.0,79,211
2024-01-01T00:46:51.000,2024-01-01T00:52:57.000,6.1,2024-01-01,0,2024,1,0.8,7.9,16.1,211,148
2024-01-01T00:54:08.000,2024-01-01T01:26:31.000,32.38333333333333,2024-01-01,0,2024,1,4.7,29.6,41.5,148,141
2024-01-01T00:49:44.000,2024-01-01T01:15:47.000,26.05,2024-01-01,0,2024,1,10.82,45.7,64.95,138,181
2024-01-01T00:30:40.000,2024-01-01T00:58:40.000,28.0,2024-01-01,0,2024,1,3.0,25.4,30.4,246,231
2024-01-01T00:26:01.000,2024-01-01T00:54:12.000,28.183333333333334,2024-01-01,0,2024,1,5.44,31.0,36.0,161,261
2024-01-01T00:28:08.000,2024-01-01T00:29:16.000,1.1333333333333333,2024-01-01,0,2024,1,0.04,3.0,8.0,113,113


In [0]:
# Enrich Silver taxi data with pickup and dropoff zones

df_zones_pickup = (
    df_zones_bronze
    .select(
        F.col("LocationID").alias("PULocationID"),
        F.col("Borough").alias("pickup_borough"),
        F.col("Zone").alias("pickup_zone"),
        F.col("service_zone").alias("pickup_service_zone")
    )
)

df_zones_dropoff = (
    df_zones_bronze
    .select(
        F.col("LocationID").alias("DOLocationID"),
        F.col("Borough").alias("dropoff_borough"),
        F.col("Zone").alias("dropoff_zone"),
        F.col("service_zone").alias("dropoff_service_zone")
    )
)

df_taxi_silver_enriched = (
    spark.table("workspace.default.silver_yellow_taxi_trips")
    .join(df_zones_pickup, on="PULocationID", how="left")
    .join(df_zones_dropoff, on="DOLocationID", how="left")
)

print("Silver enriched dataframe created.")
print("Rows:", df_taxi_silver_enriched.count())

Silver enriched dataframe created.
Rows: 2869714


In [0]:
# Preview enriched Silver data

display(
    df_taxi_silver_enriched.select(
        "pickup_date",
        "pickup_hour",
        "trip_distance",
        "trip_duration_minutes",
        "fare_amount",
        "total_amount",
        "PULocationID",
        "pickup_borough",
        "pickup_zone",
        "DOLocationID",
        "dropoff_borough",
        "dropoff_zone"
    ).limit(10)
)

pickup_date,pickup_hour,trip_distance,trip_duration_minutes,fare_amount,total_amount,PULocationID,pickup_borough,pickup_zone,DOLocationID,dropoff_borough,dropoff_zone
2024-01-01,0,1.72,19.8,17.7,22.7,186,Manhattan,Penn Station/Madison Sq West,79,Manhattan,East Village
2024-01-01,0,1.8,6.6,10.0,18.75,140,Manhattan,Lenox Hill East,236,Manhattan,Upper East Side North
2024-01-01,0,4.7,17.916666666666668,23.3,31.3,236,Manhattan,Upper East Side North,79,Manhattan,East Village
2024-01-01,0,1.4,8.3,10.0,17.0,79,Manhattan,East Village,211,Manhattan,SoHo
2024-01-01,0,0.8,6.1,7.9,16.1,211,Manhattan,SoHo,148,Manhattan,Lower East Side
2024-01-01,0,4.7,32.38333333333333,29.6,41.5,148,Manhattan,Lower East Side,141,Manhattan,Lenox Hill West
2024-01-01,0,10.82,26.05,45.7,64.95,138,Queens,LaGuardia Airport,181,Brooklyn,Park Slope
2024-01-01,0,3.0,28.0,25.4,30.4,246,Manhattan,West Chelsea/Hudson Yards,231,Manhattan,TriBeCa/Civic Center
2024-01-01,0,5.44,28.183333333333334,31.0,36.0,161,Manhattan,Midtown Center,261,Manhattan,World Trade Center
2024-01-01,0,0.04,1.1333333333333333,3.0,8.0,113,Manhattan,Greenwich Village North,113,Manhattan,Greenwich Village North


In [0]:
# Save enriched Silver Delta table

silver_enriched_table = "workspace.default.silver_yellow_taxi_trips_enriched"

df_taxi_silver_enriched.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(silver_enriched_table)

print("Enriched Silver table created successfully.")

Enriched Silver table created successfully.


In [0]:
# Validate enriched Silver table

silver_enriched_count = spark.table("workspace.default.silver_yellow_taxi_trips_enriched").count()

print("Enriched Silver rows:", silver_enriched_count)

Enriched Silver rows: 2869714


In [0]:
# Validate enriched Silver Delta table

silver_enriched_count = spark.table(
    "workspace.default.silver_yellow_taxi_trips_enriched"
).count()

print("Enriched Silver rows:", silver_enriched_count)

display(
    spark.table("workspace.default.silver_yellow_taxi_trips_enriched")
    .select(
        "pickup_date",
        "pickup_hour",
        "trip_distance",
        "trip_duration_minutes",
        "fare_amount",
        "total_amount",
        "pickup_borough",
        "pickup_zone",
        "dropoff_borough",
        "dropoff_zone"
    )
    .limit(10)
)

Enriched Silver rows: 2869714


pickup_date,pickup_hour,trip_distance,trip_duration_minutes,fare_amount,total_amount,pickup_borough,pickup_zone,dropoff_borough,dropoff_zone
2024-01-01,0,1.72,19.8,17.7,22.7,Manhattan,Penn Station/Madison Sq West,Manhattan,East Village
2024-01-01,0,1.8,6.6,10.0,18.75,Manhattan,Lenox Hill East,Manhattan,Upper East Side North
2024-01-01,0,4.7,17.916666666666668,23.3,31.3,Manhattan,Upper East Side North,Manhattan,East Village
2024-01-01,0,1.4,8.3,10.0,17.0,Manhattan,East Village,Manhattan,SoHo
2024-01-01,0,0.8,6.1,7.9,16.1,Manhattan,SoHo,Manhattan,Lower East Side
2024-01-01,0,4.7,32.38333333333333,29.6,41.5,Manhattan,Lower East Side,Manhattan,Lenox Hill West
2024-01-01,0,10.82,26.05,45.7,64.95,Queens,LaGuardia Airport,Brooklyn,Park Slope
2024-01-01,0,3.0,28.0,25.4,30.4,Manhattan,West Chelsea/Hudson Yards,Manhattan,TriBeCa/Civic Center
2024-01-01,0,5.44,28.183333333333334,31.0,36.0,Manhattan,Midtown Center,Manhattan,World Trade Center
2024-01-01,0,0.04,1.1333333333333333,3.0,8.0,Manhattan,Greenwich Village North,Manhattan,Greenwich Village North
